# 33. SegFormer가 등장한 배경

5장은 Transformer 기반 semantic segmentation 모델인 **SegFormer**를 중심으로 구성합니다. 4장에서 ViT, DETR, Swin, SAM까지 살펴봤다면, 이제 Transformer를 segmentation에 어떻게 실용적으로 적용하는지 집중해서 봅니다.

SegFormer는 단순히 `ViT를 segmentation에 붙인 모델`이 아닙니다. segmentation에 필요한 multi-scale feature, dense prediction, 효율적인 attention, 가벼운 decoder를 함께 고려한 구조입니다.

이번 노트북의 목표는 다음과 같습니다.

- ViT에서 SegFormer로 넘어가는 맥락을 이해합니다.
- semantic segmentation 모델에 필요한 조건을 정리합니다.
- SegFormer의 핵심 구성인 MiT encoder와 MLP decoder의 위치를 파악합니다.
- 5장에서 다룰 전체 흐름을 잡습니다.

## 33-1. 준비

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False

## 33-2. 지금까지의 흐름

SegFormer는 3장의 segmentation과 4장의 Transformer가 만나는 지점에 있습니다.

```text
U-Net / DeepLabV3
  -> dense prediction과 multi-scale feature의 필요성

ViT / Swin Transformer
  -> 이미지를 token으로 보고 attention으로 관계를 모델링

SegFormer
  -> Transformer encoder + lightweight segmentation decoder
```

In [ ]:
items = [
    ('CNN 분류', 'ResNet'),
    ('CNN segmentation', 'U-Net / DeepLabV3'),
    ('Vision Transformer', 'ViT'),
    ('계층적 Transformer', 'Swin'),
    ('Transformer segmentation', 'SegFormer'),
]

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
for i, (title, model) in enumerate(items):
    x = i * 2.1
    ax.add_patch(plt.Rectangle((x, 0.8), 1.55, 0.8, facecolor='#dbeafe', edgecolor='#2563eb', linewidth=1.8))
    ax.text(x + 0.775, 1.32, title, ha='center', va='center', weight='bold', fontsize=9)
    ax.text(x + 0.775, 1.03, model, ha='center', va='center', fontsize=9)
    if i < len(items) - 1:
        ax.annotate('', xy=(x + 2.0, 1.2), xytext=(x + 1.58, 1.2), arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-0.2, 9.8)
ax.set_ylim(0.4, 1.9)
ax.set_title('SegFormer로 이어지는 커리큘럼 흐름')
plt.show()

## 33-3. Segmentation 모델에 필요한 조건

이미지 분류 모델은 이미지 전체를 하나의 class로 요약하면 됩니다. 하지만 semantic segmentation은 각 픽셀 또는 각 위치마다 class를 예측해야 합니다.

따라서 좋은 segmentation backbone은 다음 조건을 만족해야 합니다.

- 작은 물체와 큰 물체를 모두 다룰 수 있는 multi-scale feature가 필요합니다.
- 위치 정보가 너무 빨리 사라지면 안 됩니다.
- 고해상도 feature를 처리할 계산 효율이 필요합니다.
- decoder는 feature를 pixel-level prediction으로 복원해야 합니다.

In [ ]:
requirements = ['multi-scale', 'location', 'efficiency', 'simple decoder']
scores = {
    'plain ViT': [1, 2, 1, 2],
    'Swin': [4, 4, 4, 3],
    'SegFormer': [4, 4, 4, 5],
}

x = np.arange(len(requirements))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 4))
for i, (name, values) in enumerate(scores.items()):
    ax.bar(x + (i - 1) * width, values, width, label=name)
ax.set_xticks(x)
ax.set_xticklabels(requirements)
ax.set_ylim(0, 5.5)
ax.set_ylabel('conceptual fit')
ax.set_title('Segmentation 관점에서 본 구조 적합도')
ax.legend()
plt.show()

## 33-4. SegFormer의 핵심 아이디어

SegFormer는 크게 두 부분으로 나눌 수 있습니다.

```text
image
  -> MiT encoder
     -> multi-scale Transformer features
  -> lightweight MLP decoder
     -> segmentation logits
  -> upsample
     -> pixel-wise class map
```

MiT는 Mix Transformer encoder를 의미합니다. SegFormer는 positional encoding에 강하게 의존하지 않고, overlapping patch embedding과 efficient self-attention을 활용합니다. Decoder는 복잡한 convolution decoder가 아니라 각 stage feature를 MLP로 맞춘 뒤 결합하는 단순한 구조입니다.

## 정리

- ViT는 이미지를 token sequence로 보는 관점을 열었지만, segmentation에는 dense prediction과 multi-scale feature가 추가로 필요합니다.
- SegFormer는 Transformer encoder를 segmentation에 맞게 계층적으로 구성하고, 가벼운 MLP decoder로 mask를 예측합니다.
- 5장의 핵심 흐름은 `ViT의 한계 -> MiT encoder -> efficient attention -> MLP decoder -> SegFormer forward -> 실습`입니다.

다음 노트북 `34_ViT를_Segmentation에_그대로_쓰기_어려운_이유.ipynb`에서는 ViT와 dense prediction 사이의 간극을 더 구체적으로 살펴봅니다.